# Relatório de Performance Completo

Este notebook valida:
- **Treino Robusto**: Generalista treinado com dados até 2023
- **Modelos Especialistas**: Fine-tuning específico por ativo

---

## Estrutura

### PARTE 1: Análise do Modelo Robusto (Generalista)
1. Configuração do Ambiente
2. Funções Auxiliares de Carregamento
3. Diagnóstico do Treinamento
4. Carregamento do Modelo Generalista
5. Validação Global 2024

### PARTE 2: Análise dos Especialistas (Fine-Tuning)
6. Comparativo Multi-Modelo
7. Visualização Avançada (Backtest)
8. Conclusão Estatística


---

## PARTE 1: Análise do Modelo Robusto (Generalista)


### Célula 1: Configuração do Ambiente


In [ ]:
# Imports padrão
import sys
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from torch.utils.data import DataLoader, TensorDataset
import json

# Adiciona raiz ao path
sys.path.append(os.path.abspath(''))

# Imports do projeto
from src.config import MODEL_CONFIG, DATA_CONFIG, PATHS
from src.model import DeepHestonHybrid
from src.data_loader import carregar_taxa_juros, _calcular_features_ativo, calcular_peso_amostra
from src.physics import heston_residual

# Configurações visuais
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Ambiente configurado. Dispositivo: {device}')


### Célula 2: Funções Auxiliares de Carregamento


In [ ]:
def criar_dataset_test_2024(caminho_pasta_opcoes: str, df_juros: pd.DataFrame, seq_length: int, stats: dict, ativo_alvo: str = None):
    """
    Gera dataset de teste filtrando pelo ativo específico.
    
    Args:
        caminho_pasta_opcoes: Caminho para pasta com dados de 2024
        df_juros: DataFrame com taxas de juros
        seq_length: Tamanho da sequência LSTM
        stats: Estatísticas de normalização do treino
        ativo_alvo: Código do ativo (opcional, None = todos)
    
    Returns:
        TensorDataset com sequências e features normalizadas
    """
    lista_dfs = []
    if not os.path.exists(caminho_pasta_opcoes):
        print(f'ERRO: Pasta não encontrada: {caminho_pasta_opcoes}')
        return None

    print(f'Procurando arquivos de {ativo_alvo if ativo_alvo else "TODOS"} em: {caminho_pasta_opcoes}')
    nome_arquivo_selic = 'taxa_selic.csv'

    for arquivo in os.listdir(caminho_pasta_opcoes):
        # Filtro por ativo se especificado
        if ativo_alvo and ativo_alvo not in arquivo:
            continue

        if arquivo.endswith('.csv') and arquivo != nome_arquivo_selic:
            path_completo = os.path.join(caminho_pasta_opcoes, arquivo)
            try:
                df_head = pd.read_csv(path_completo, nrows=2)
                if 'time' not in df_head.columns:
                    continue
                
                df_temp = pd.read_csv(path_completo)
                df_temp['time'] = pd.to_datetime(df_temp['time'], errors='coerce', utc=True).dt.tz_localize(None)
                
                if 'ativo' not in df_temp.columns:
                    simbolo = arquivo.split('_')[0] if '_' in arquivo else 'UNKNOWN'
                    df_temp['ativo'] = simbolo
                
                cols_req = ['time', 'spot_price', 'strike', 'premium', 'days_to_maturity', 'ativo']
                if all(c in df_temp.columns for c in cols_req):
                    lista_dfs.append(df_temp[cols_req])
                    print(f'Carregado: {arquivo}')
            except Exception as e:
                continue
    
    if not lista_dfs:
        print(f'Nenhum arquivo encontrado para o ativo {ativo_alvo}.')
        return None

    df_full = pd.concat(lista_dfs, ignore_index=True)
    
    # Filtro 2024
    df_full = df_full[df_full['time'].dt.year == 2024]
    
    if 'premium' in df_full.columns:
        df_full = df_full[df_full['premium'] > 0.0]
    
    if df_full.empty:
        print('ERRO: Dataset vazio após filtro de ano 2024!')
        return None
    
    print(f'Dados 2024 carregados: {len(df_full)} registros.')

    # Merge Juros
    df_full['data_only'] = df_full['time'].dt.normalize()
    if df_juros is not None:
        df_full = pd.merge(df_full, df_juros, on='data_only', how='left')
        df_full = df_full.sort_values('time')
        df_full['r'] = df_full['r'].ffill().bfill().fillna(0.10)
    else:
        df_full['r'] = 0.10

    sequences, pinn_inputs, targets, timestamps = [], [], [], []
    
    # Gerar Sequências
    for _, df_grupo in df_full.groupby('ativo'):
        df_grupo = df_grupo.sort_values('time')
        df_asset = df_grupo[['time', 'spot_price']].drop_duplicates('time').copy()
        
        if len(df_asset) < seq_length + 20:
            continue
        
        df_asset = _calcular_features_ativo(df_asset)
        asset_dates = pd.to_datetime(df_asset['time']).tolist()
        asset_feats = df_asset[['log_ret', 'rolling_vol_20']].values.astype(np.float32)
        date_map = {ts: i for i, ts in enumerate(asset_dates)}
        
        for row in df_grupo.itertuples():
            idx = date_map.get(row.time)
            if idx is not None and idx >= seq_length:
                sequences.append(asset_feats[idx-seq_length : idx])
                pinn_inputs.append([row.spot_price, row.strike, row.days_to_maturity/365.25, row.r])
                targets.append(row.premium)
                timestamps.append(row.time.timestamp())

    if not targets:
        return None

    # Conversão para Tensores
    X_seq = np.array(sequences, dtype=np.float32)
    X_phy = np.array(pinn_inputs, dtype=np.float32)
    y = np.array(targets, dtype=np.float32).reshape(-1, 1)
    
    # Normalizar Target (Preço / Strike Real)
    y_norm = y / (X_phy[:, 1:2] + 1e-8)
    
    # Normalizar Inputs Físicos usando Stats do Treino
    X_phy[:, 0] = (X_phy[:, 0] - stats['S_min']) / (stats['S_max'] - stats['S_min'])
    X_phy[:, 1] = (X_phy[:, 1] - stats['K_min']) / (stats['K_max'] - stats['K_min'])
    X_phy[:, 2] = X_phy[:, 2] / stats['T_max']
    
    X_time = np.array(timestamps, dtype=np.float64).reshape(-1, 1)
    weights = np.ones_like(y_norm)
    
    return TensorDataset(
        torch.from_numpy(X_seq),
        torch.from_numpy(X_phy),
        torch.from_numpy(y_norm),
        torch.from_numpy(X_time),
        torch.from_numpy(weights)
    )


### Célula 3: Diagnóstico do Treinamento (Histórico)


In [ ]:
# Carregar histórico de treino
history_path = os.path.join(PATHS['results_dir'], 'training_history.csv')

if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    
    # Plot 1: Curvas de Loss (Train vs Validação)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.semilogy(df_history['epoch'], df_history['train_loss'], label='Train Loss', color='blue')
    ax1.semilogy(df_history['epoch'], df_history['val_loss'], label='Val Loss', color='red')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss (log scale)')
    ax1.set_title('Convergência: Train vs Validação')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Evolução dos pesos da física
    if 'weight_pde' in df_history.columns:
        ax2.plot(df_history['epoch'], df_history['weight_pde'], color='green')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Peso PDE')
        ax2.set_title('Curriculum Learning: Evolução do Peso da Física')
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Plot 3: L_PDE vs L_Data
    if 'L_PDE' in df_history.columns and 'L_Data' in df_history.columns:
        plt.figure(figsize=(8, 6))
        plt.scatter(df_history['L_Data'], df_history['L_PDE'], alpha=0.5, c=df_history['epoch'], cmap='viridis')
        plt.xlabel('Loss Dados')
        plt.ylabel('Loss Física (PDE)')
        plt.title('Dispersão: Física vs Dados')
        plt.colorbar(label='Epoch')
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print(f'AVISO: Arquivo de histórico não encontrado em {history_path}')


### Célula 4: Carregamento do Modelo Generalista


In [ ]:
# Carregar data_stats
stats_path = os.path.join(PATHS['model_save_dir'], 'data_stats.json')
with open(stats_path, 'r') as f:
    train_stats = json.load(f)

print('Estatísticas de Treino Carregadas:', train_stats)

# Inicializar Modelo Generalista
model_generalista = DeepHestonHybrid(config=MODEL_CONFIG, data_stats=train_stats)
model_path = os.path.join(PATHS['model_save_dir'], 'best_hybrid_model.pth')

if not os.path.exists(model_path):
    # Fallback para best_model_weights.pth se não existir
    model_path = os.path.join(PATHS['model_save_dir'], 'best_model_weights.pth')

model_generalista.load_state_dict(torch.load(model_path, map_location=device))
model_generalista.to(device)
model_generalista.eval()

print('Modelo Generalista Carregado.')


### Célula 5: Validação Global 2024


In [ ]:
# Carregar dados de 2024 (ex: PETR4)
path_databricks_2024 = os.path.join(PATHS['raw_data'], 'Databricks', '2024')
df_selic = carregar_taxa_juros(PATHS['selic_data'])

ds_test = criar_dataset_test_2024(
    path_databricks_2024,
    df_selic,
    DATA_CONFIG['sequence_length'],
    train_stats,
    ativo_alvo='PETR4'  # Altere conforme necessário
)

if ds_test is None:
    print('ERRO: Não foi possível carregar dados de teste 2024.')
else:
    loader = DataLoader(ds_test, batch_size=4096, shuffle=False)
    
    res_real, res_pred, res_time, res_strike, res_spot = [], [], [], [], []
    
    with torch.no_grad():
        for batch in loader:
            x_seq = batch[0].to(device)
            x_phy_raw = batch[1].to(device)
            y_norm = batch[2].to(device)
            times = batch[3]
            
            # Normalização Manual usando stats do TREINO
            S_raw = x_phy_raw[:, 0:1]
            K_raw = x_phy_raw[:, 1:2]
            T_raw = x_phy_raw[:, 2:3]
            r_raw = x_phy_raw[:, 3:4]
            
            S_norm = (S_raw - train_stats['S_min']) / (train_stats['S_max'] - train_stats['S_min'])
            K_norm = (K_raw - train_stats['K_min']) / (train_stats['K_max'] - train_stats['K_min'])
            T_norm = T_raw / train_stats['T_max']
            
            x_phy_norm = torch.cat([S_norm, K_norm, T_norm, r_raw], dim=1)
            
            # Previsão
            out = model_generalista(x_seq, x_phy_norm)
            
            # Desnormalizar Preço (Pred * Strike)
            price_pred = out['price'] * K_raw
            price_real = y_norm * K_raw
            
            res_real.extend(price_real.cpu().numpy().flatten())
            res_pred.extend(price_pred.cpu().numpy().flatten())
            res_time.extend(times.numpy().flatten())
            res_strike.extend(K_raw.cpu().numpy().flatten())
            res_spot.extend(S_raw.cpu().numpy().flatten())
    
    # Criar DataFrame de Resultados
    df_2024 = pd.DataFrame({
        'timestamp': res_time,
        'Real': res_real,
        'Previsto': res_pred,
        'Strike': res_strike,
        'Spot': res_spot
    })
    df_2024['Data'] = pd.to_datetime(df_2024['timestamp'], unit='s')
    df_2024['Moneyness'] = df_2024['Spot'] / df_2024['Strike']
    
    # Métricas
    rmse_generalista = np.sqrt(mean_squared_error(df_2024['Real'], df_2024['Previsto']))
    mae_generalista = mean_absolute_error(df_2024['Real'], df_2024['Previsto'])
    r2_generalista = r2_score(df_2024['Real'], df_2024['Previsto'])
    
    print(f'=== RESULTADOS GENERALISTA 2024 ===')
    print(f'RMSE: R$ {rmse_generalista:.4f}')
    print(f'MAE: R$ {mae_generalista:.4f}')
    print(f'R² Score: {r2_generalista:.4f}')
    
    # Visualização 3D: Superfície de Preço Prevista
    # Gerar grid sintético
    S_grid = np.linspace(0.6, 1.4, 30)
    T_grid = np.linspace(0.1, 1.0, 30)
    S_mesh, T_mesh = np.meshgrid(S_grid, T_grid)
    
    # Criar inputs sintéticos
    n_points = S_mesh.size
    S_flat = S_mesh.flatten()
    T_flat = T_mesh.flatten()
    K_flat = np.ones_like(S_flat)  # K=1 para normalização
    r_flat = np.ones_like(S_flat) * 0.10
    
    # Criar sequências dummy (zeros)
    seq_dummy = np.zeros((n_points, DATA_CONFIG['sequence_length'], 2), dtype=np.float32)
    
    # Normalizar inputs
    S_norm_flat = (S_flat - train_stats['S_min']) / (train_stats['S_max'] - train_stats['S_min'])
    K_norm_flat = (K_flat - train_stats['K_min']) / (train_stats['K_max'] - train_stats['K_min'])
    T_norm_flat = T_flat / train_stats['T_max']
    
    x_phy_grid = np.column_stack([S_norm_flat, K_norm_flat, T_norm_flat, r_flat])
    
    # Converter para tensores
    seq_tensor = torch.from_numpy(seq_dummy).to(device)
    phy_tensor = torch.from_numpy(x_phy_grid.astype(np.float32)).to(device)
    
    # Previsão
    with torch.no_grad():
        out_grid = model_generalista(seq_tensor, phy_tensor)
        price_grid = out_grid['price'].cpu().numpy().reshape(30, 30)
    
    # Calcular Resíduo da EDP
    with torch.no_grad():
        # Habilitar gradientes temporariamente
        phy_tensor_grad = phy_tensor.clone().requires_grad_(True)
        out_pde = model_generalista(seq_tensor, phy_tensor_grad)
        residual = heston_residual(
            out_pde['price'],
            phy_tensor_grad,
            out_pde['nu'],
            {},  # params vazio, modelo aprende
            return_pointwise=True
        )
        residual_grid = residual.cpu().numpy().reshape(30, 30)
    
    # Plot 3D: Superfície de Preço
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure(figsize=(14, 6))
    
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(S_mesh, T_mesh, price_grid, cmap='viridis', alpha=0.7)
    ax1.set_xlabel('S (Spot/Strike)')
    ax1.set_ylabel('T (Anos)')
    ax1.set_zlabel('Preço')
    ax1.set_title('Superfície de Preço Prevista')
    
    # Plot 3D: Superfície de Resíduo
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.plot_surface(S_mesh, T_mesh, residual_grid, cmap='coolwarm', alpha=0.7)
    ax2.set_xlabel('S (Spot/Strike)')
    ax2.set_ylabel('T (Anos)')
    ax2.set_zlabel('Resíduo PDE')
    ax2.set_title('Superfície de Resíduo da EDP')
    
    plt.tight_layout()
    plt.show()


---

## PARTE 2: Análise dos Especialistas (Fine-Tuning)


### Célula 6: Comparativo Multi-Modelo


In [ ]:
# Lista de ativos para análise
ativos = ['PETR4', 'VALE3', 'BOVA11']

resultados = []

for ativo in ativos:
    print(f'\n=== Analisando {ativo} ===')
    
    # 1. Carregar dados 2024 específicos do ativo
    ds_test_ativo = criar_dataset_test_2024(
        path_databricks_2024,
        df_selic,
        DATA_CONFIG['sequence_length'],
        train_stats,
        ativo_alvo=ativo
    )
    
    if ds_test_ativo is None:
        print(f'Dados não encontrados para {ativo}, pulando...')
        continue
    
    loader_ativo = DataLoader(ds_test_ativo, batch_size=4096, shuffle=False)
    
    # 2. Previsões com Modelo Generalista
    pred_gen, real = [], []
    
    with torch.no_grad():
        for batch in loader_ativo:
            x_seq = batch[0].to(device)
            x_phy_raw = batch[1].to(device)
            y_norm = batch[2].to(device)
            
            # Normalizar
            S_raw, K_raw, T_raw, r_raw = x_phy_raw[:, 0:1], x_phy_raw[:, 1:2], x_phy_raw[:, 2:3], x_phy_raw[:, 3:4]
            S_norm = (S_raw - train_stats['S_min']) / (train_stats['S_max'] - train_stats['S_min'])
            K_norm = (K_raw - train_stats['K_min']) / (train_stats['K_max'] - train_stats['K_min'])
            T_norm = T_raw / train_stats['T_max']
            x_phy_norm = torch.cat([S_norm, K_norm, T_norm, r_raw], dim=1)
            
            # Previsão Generalista
            out_gen = model_generalista(x_seq, x_phy_norm)
            price_gen = out_gen['price'] * K_raw
            price_real = y_norm * K_raw
            
            pred_gen.extend(price_gen.cpu().numpy().flatten())
            real.extend(price_real.cpu().numpy().flatten())
    
    mae_gen = mean_absolute_error(real, pred_gen)
    
    # 3. Carregar Modelo Especialista
    # Extrair base do ativo (ex: PETR4 -> PETR)
    ativo_base = ativo[:4] if len(ativo) == 5 else ativo
    model_especialista_path = os.path.join(PATHS['model_save_dir'], f'best_{ativo_base}.pt')
    
    if not os.path.exists(model_especialista_path):
        # Tentar com .pth
        model_especialista_path = os.path.join(PATHS['model_save_dir'], f'best_{ativo_base}.pth')
    
    if os.path.exists(model_especialista_path):
        model_especialista = DeepHestonHybrid(config=MODEL_CONFIG, data_stats=train_stats)
        model_especialista.load_state_dict(torch.load(model_especialista_path, map_location=device), strict=False)
        model_especialista.to(device)
        model_especialista.eval()
        
        # 4. Previsões com Modelo Especialista
        pred_espec = []
        
        with torch.no_grad():
            for batch in loader_ativo:
                x_seq = batch[0].to(device)
                x_phy_raw = batch[1].to(device)
                
                # Normalizar
                S_raw, K_raw, T_raw, r_raw = x_phy_raw[:, 0:1], x_phy_raw[:, 1:2], x_phy_raw[:, 2:3], x_phy_raw[:, 3:4]
                S_norm = (S_raw - train_stats['S_min']) / (train_stats['S_max'] - train_stats['S_min'])
                K_norm = (K_raw - train_stats['K_min']) / (train_stats['K_max'] - train_stats['K_min'])
                T_norm = T_raw / train_stats['T_max']
                x_phy_norm = torch.cat([S_norm, K_norm, T_norm, r_raw], dim=1)
                
                # Previsão Especialista
                out_espec = model_especialista(x_seq, x_phy_norm)
                price_espec = out_espec['price'] * K_raw
                
                pred_espec.extend(price_espec.cpu().numpy().flatten())
        
        mae_espec = mean_absolute_error(real, pred_espec)
        
        # 5. Calcular ganho percentual
        ganho = ((mae_gen - mae_espec) / mae_gen) * 100
        
        resultados.append({
            'Ativo': ativo,
            'MAE Generalista': mae_gen,
            'MAE Especialista': mae_espec,
            'Ganho (%)': ganho
        })
        
        print(f'MAE Generalista: R$ {mae_gen:.4f}')
        print(f'MAE Especialista: R$ {mae_espec:.4f}')
        print(f'Ganho: {ganho:.2f}%')
    else:
        print(f'Modelo especialista não encontrado em {model_especialista_path}')
        resultados.append({
            'Ativo': ativo,
            'MAE Generalista': mae_gen,
            'MAE Especialista': None,
            'Ganho (%)': None
        })

# Criar DataFrame de resultados
df_resultados = pd.DataFrame(resultados)
print('\n=== RESUMO COMPARATIVO ===')
print(df_resultados.to_string(index=False))


### Célula 7: Visualização Avançada (Backtest)


In [ ]:
# Painel de 3 Gráficos por Ativo

for ativo in ativos:
    print(f'\nGerando visualizações para {ativo}...')
    
    # Carregar dados novamente
    ds_test_ativo = criar_dataset_test_2024(
        path_databricks_2024,
        df_selic,
        DATA_CONFIG['sequence_length'],
        train_stats,
        ativo_alvo=ativo
    )
    
    if ds_test_ativo is None:
        continue
    
    loader_ativo = DataLoader(ds_test_ativo, batch_size=4096, shuffle=False)
    
    # Previsões
    pred_gen, pred_espec, real, timestamps = [], [], [], []
    nu_espec = []  # Volatilidade latente do especialista
    
    with torch.no_grad():
        for batch in loader_ativo:
            x_seq = batch[0].to(device)
            x_phy_raw = batch[1].to(device)
            y_norm = batch[2].to(device)
            times = batch[3]
            
            S_raw, K_raw, T_raw, r_raw = x_phy_raw[:, 0:1], x_phy_raw[:, 1:2], x_phy_raw[:, 2:3], x_phy_raw[:, 3:4]
            S_norm = (S_raw - train_stats['S_min']) / (train_stats['S_max'] - train_stats['S_min'])
            K_norm = (K_raw - train_stats['K_min']) / (train_stats['K_max'] - train_stats['K_min'])
            T_norm = T_raw / train_stats['T_max']
            x_phy_norm = torch.cat([S_norm, K_norm, T_norm, r_raw], dim=1)
            
            # Generalista
            out_gen = model_generalista(x_seq, x_phy_norm)
            price_gen = out_gen['price'] * K_raw
            
            # Especialista (se existir)
            ativo_base = ativo[:4] if len(ativo) == 5 else ativo
            model_especialista_path = os.path.join(PATHS['model_save_dir'], f'best_{ativo_base}.pt')
            
            if not os.path.exists(model_especialista_path):
                model_especialista_path = os.path.join(PATHS['model_save_dir'], f'best_{ativo_base}.pth')
            
            if os.path.exists(model_especialista_path):
                model_especialista = DeepHestonHybrid(config=MODEL_CONFIG, data_stats=train_stats)
                model_especialista.load_state_dict(torch.load(model_especialista_path, map_location=device), strict=False)
                model_especialista.to(device)
                model_especialista.eval()
                
                out_espec = model_especialista(x_seq, x_phy_norm)
                price_espec = out_espec['price'] * K_raw
                
                pred_espec.extend(price_espec.cpu().numpy().flatten())
                nu_espec.extend(out_espec['nu'].cpu().numpy().flatten())
            else:
                pred_espec = None
                nu_espec = None
            
            price_real = y_norm * K_raw
            
            pred_gen.extend(price_gen.cpu().numpy().flatten())
            real.extend(price_real.cpu().numpy().flatten())
            timestamps.extend(times.numpy().flatten())
    
    # Criar DataFrame
    df_vis = pd.DataFrame({
        'timestamp': timestamps,
        'Real': real,
        'Generalista': pred_gen
    })
    
    if pred_espec is not None:
        df_vis['Especialista'] = pred_espec
        df_vis['Volatilidade'] = np.sqrt(nu_espec)
    
    df_vis['Data'] = pd.to_datetime(df_vis['timestamp'], unit='s')
    df_vis = df_vis.sort_values('Data')
    
    # Painel de Gráficos
    if pred_espec is not None:
        fig, axes = plt.subplots(3, 1, figsize=(14, 12))
        
        # 1. Série Temporal
        axes[0].plot(df_vis['Data'], df_vis['Real'], label='Real', color='black', linewidth=1)
        axes[0].plot(df_vis['Data'], df_vis['Generalista'], label='Generalista', color='blue', linestyle='--', alpha=0.7)
        axes[0].plot(df_vis['Data'], df_vis['Especialista'], label='Especialista', color='green', linewidth=1.5)
        axes[0].set_title(f'{ativo}: Real vs Previsões (2024)')
        axes[0].set_xlabel('Data')
        axes[0].set_ylabel('Preço')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. Scatter Plot
        axes[1].scatter(df_vis['Real'], df_vis['Generalista'], alpha=0.3, label='Generalista', color='blue', s=10)
        axes[1].scatter(df_vis['Real'], df_vis['Especialista'], alpha=0.3, label='Especialista', color='green', s=10)
        axes[1].plot([df_vis['Real'].min(), df_vis['Real'].max()],
                     [df_vis['Real'].min(), df_vis['Real'].max()], 'k--', label='Perfeito')
        axes[1].set_title(f'{ativo}: Real vs Previsto')
        axes[1].set_xlabel('Preço Real')
        axes[1].set_ylabel('Preço Previsto')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # 3. Volatilidade Latente
        axes[2].plot(df_vis['Data'], df_vis['Volatilidade'], color='orange', linewidth=1)
        axes[2].set_title(f'{ativo}: Volatilidade Latente (Especialista)')
        axes[2].set_xlabel('Data')
        axes[2].set_ylabel('sqrt(nu_t)')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print(f'Modelo especialista não encontrado para {ativo}, pulando visualizações...')


### Célula 8: Conclusão Estatística


In [ ]:
# Tabela Final Sumarizada

print('\n' + '='*60)
print('CONCLUSÃO: GANHO DO FINE-TUNING')
print('='*60)

if not df_resultados.empty:
    print(df_resultados.to_string(index=False))
    
    # Estatísticas Agregadas
    ganhos_validos = df_resultados[df_resultados['Ganho (%)'].notna()]['Ganho (%)']
    
    if len(ganhos_validos) > 0:
        media_ganho = ganhos_validos.mean()
        mediana_ganho = ganhos_validos.median()
        
        print(f'\nMédia de Ganho: {media_ganho:.2f}%')
        print(f'Mediana de Ganho: {mediana_ganho:.2f}%')
        
        if media_ganho > 0:
            print('\n✅ Fine-tuning MELHOROU a performance em média.')
        else:
            print('\n⚠️ Fine-tuning NÃO trouxe melhorias significativas.')
    else:
        print('\nNenhum modelo especialista encontrado para comparação.')
else:
    print('Nenhum resultado disponível.')

print('\n' + '='*60)
print('FIM DO RELATÓRIO')
print('='*60)
